# EDA: Hai cỗ máy bên trong - Mùa vụ khoẻ, Lợi nhuận hỏng

**Datathon 2026 — The Gridbreaker | Phần 2**

## Tóm tắt

Doanh nghiệp thời trang TMĐT này có hai cơ chế hoạt động song song. Cỗ máy mùa vụ vẫn vận hành đúng: Quý 2 đóng góp 36% doanh thu nhưng 45% lợi nhuận năm, tháng 5 tạo doanh thu ngày gấp 2.6 lần tháng 12. Cỗ máy lợi nhuận đã hỏng: doanh thu đạt đỉnh 2.10 tỷ VND năm 2016, sập 39% xuống 1.14 tỷ năm 2019, không hồi phục. Acquisition và web traffic tăng đều qua các năm, loại bỏ giả thuyết cạn đầu phễu.

Phân tích chuỗi nhân quả cho thấy: tồn kho Streetwear/Outdoor đã leo từ ~150 ngày (2012) lên gần 2.000 ngày DOS (2022), buộc doanh nghiệp giảm giá sâu để giải phóng kho, đẩy biên Streetwear khi khuyến mãi xuống -15.57% (lỗ luỹ kế 578.6M VND). Kiến trúc khuyến mãi củng cố vòng xoáy này: 80% chương trình áp dụng toàn sàn, kéo Streetwear vượt ngưỡng breakeven 20% discount.

Bài này đề xuất 4 hành động định lượng được, ước tính giải phóng 471M VND vốn lưu động và phục hồi biên Streetwear lên 4-6% trong 12 tháng.

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Print-safe palette (tested grayscale)
COLORS = {
    'navy':       '#1F3A68',
    'gold':       '#C9A227',
    'brick':      '#A03030',
    'forest':     '#2D7D40',
    'orange':     '#D88C26',
    'grey':       '#666666',
    'lightgrey':  '#CCCCCC',
}
CAT_COLORS = {
    'Streetwear': '#A03030',
    'Outdoor':    '#D88C26',
    'GenZ':       '#2D7D40',
    'Casual':     '#1F3A68',
}

plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linewidth': 0.5,
    'figure.dpi': 100,
})

DATA = './'   # adjust to dataset path

In [ ]:
# Load all tables
sales    = pd.read_csv(f'{DATA}sales.csv', parse_dates=['Date'])
products = pd.read_csv(f'{DATA}products.csv')
oi       = pd.read_csv(f'{DATA}order_items.csv', low_memory=False)
promos   = pd.read_csv(f'{DATA}promotions.csv', parse_dates=['start_date','end_date'])
inv      = pd.read_csv(f'{DATA}inventory.csv', parse_dates=['snapshot_date'])
customers= pd.read_csv(f'{DATA}customers.csv', parse_dates=['signup_date'])
web      = pd.read_csv(f'{DATA}web_traffic.csv', parse_dates=['date'])
returns  = pd.read_csv(f'{DATA}returns.csv', parse_dates=['return_date'])
reviews  = pd.read_csv(f'{DATA}reviews.csv', parse_dates=['review_date'])
geog     = pd.read_csv(f'{DATA}geography.csv')
shipments= pd.read_csv(f'{DATA}shipments.csv', parse_dates=['ship_date','delivery_date'])
payments = pd.read_csv(f'{DATA}payments.csv')

# Quick dimension check
print(f'sales:     {sales.shape},     {sales["Date"].min().date()} to {sales["Date"].max().date()}')
print(f'products:  {products.shape}')
print(f'order_items:{oi.shape}')
print(f'promotions:{promos.shape}')
print(f'inventory: {inv.shape}')
print(f'customers: {customers.shape}')
print(f'web_traffic:{web.shape}')

---

## Act 1 - Chẩn đoán: Một cỗ máy chạy, một cỗ máy hỏng

Mục tiêu của Act 1 là xác định phạm vi vấn đề. Ba câu hỏi:

1. Doanh thu và lợi nhuận diễn biến ra sao qua 11 năm?
2. Đầu phễu (traffic, signup) có suy giảm theo không?
3. Tính mùa vụ còn nguyên vẹn hay đã rạn?

Câu trả lời sẽ định hình toàn bộ phần còn lại: nếu đầu phễu khoẻ mà doanh thu sập, vấn đề nằm ở chuyển đổi và biên lợi nhuận chứ không phải nhu cầu thị trường.

### 1.1 Quỹ đạo doanh thu, lợi nhuận và biên lợi nhuận theo năm

In [ ]:
sales['year']   = sales['Date'].dt.year
sales['profit'] = sales['Revenue'] - sales['COGS']

annual = sales.groupby('year').agg(
    revenue = ('Revenue', 'sum'),
    cogs    = ('COGS',    'sum'),
    profit  = ('profit',  'sum'),
    n_days  = ('Date',    'count')
).reset_index()
annual['margin_pct'] = annual['profit'] / annual['revenue'] * 100

# 2012 only has 181 days, exclude from yoy analysis
annual_full = annual[annual['n_days'] >= 365].copy()

print(annual.round(2).to_string(index=False))
print(f'\nPeak revenue year: {int(annual_full.loc[annual_full["revenue"].idxmax(), "year"])}')
print(f'Peak revenue: {annual_full["revenue"].max()/1e9:.2f} ty VND')
print(f'2018 to 2019 revenue change: {(annual_full[annual_full.year==2019]["revenue"].values[0]/annual_full[annual_full.year==2018]["revenue"].values[0]-1)*100:.1f}%')

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

x = annual_full['year']
ax1.plot(x, annual_full['revenue']/1e9, marker='o', color=COLORS['navy'], lw=2, label='Doanh thu')
ax1.plot(x, annual_full['profit']/1e9, marker='s', color=COLORS['gold'], lw=2, label='Loi nhuan gop')
ax1.set_xlabel('Nam')
ax1.set_ylabel('Ty VND')
ax1.set_xticks(x)

# Margin on secondary axis
ax2 = ax1.twinx()
ax2.bar(x, annual_full['margin_pct'], alpha=0.15, color=COLORS['brick'], label='Bien loi nhuan (%)')
ax2.set_ylabel('Bien loi nhuan (%)', color=COLORS['brick'])
ax2.tick_params(axis='y', labelcolor=COLORS['brick'])
ax2.grid(False)
ax2.set_ylim(0, 25)

# Annotations
ax1.axvline(2016, color=COLORS['grey'], linestyle=':', alpha=0.5)
ax1.axvline(2019, color=COLORS['brick'], linestyle=':', alpha=0.7)
ax1.annotate('Dinh 2016\n2.10 ty VND', xy=(2016, 2.10), xytext=(2014.3, 2.4),
             fontsize=9, color=COLORS['grey'],
             arrowprops=dict(arrowstyle='->', color=COLORS['grey'], alpha=0.5))
ax1.annotate('Sap 2019\n-39% YoY', xy=(2019, 1.14), xytext=(2020, 1.7),
             fontsize=9, color=COLORS['brick'],
             arrowprops=dict(arrowstyle='->', color=COLORS['brick']))

ax1.set_title('Doanh thu, loi nhuan va bien loi nhuan theo nam (2013-2022)', loc='left', fontweight='bold')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
fig.text(0.01, -0.02, 'Nguon: sales.csv (n=3833 ngay, loai 2012 do du lieu thieu)', fontsize=8, color=COLORS['grey'])
plt.tight_layout()
plt.show()

**Phát hiện 1.1:**

- Đỉnh doanh thu vào năm 2016 đạt 2.10 tỷ VND. Trong giai đoạn 2017-2018 doanh thu giảm nhẹ nhưng vẫn duy trì trên 1.85 tỷ.
- Năm 2019 chứng kiến cú sập 39% so với 2018 (1.85 tỷ xuống 1.14 tỷ). Doanh thu chưa bao giờ phục hồi về mức 2018, hiện vẫn dưới ngưỡng 1.20 tỷ.
- Biên lợi nhuận biến động mạnh hơn doanh thu: dao động 9.77% (2021) đến 16.64% (2018). Đây là đặc điểm của doanh nghiệp đang phải đánh đổi giá để giữ doanh số.

Hai câu hỏi cần trả lời tiếp: cú sập 2019 có phải do mất khách hàng tổng thể (cạn cầu)? Hay do conversion/order frequency suy giảm (vấn đề nội tại)?

### 1.2 Đầu phễu vẫn khoẻ - acquisition và traffic không sập

In [ ]:
# Customer signups by year
customers['signup_year'] = customers['signup_date'].dt.year
acq = customers.groupby('signup_year').size().reset_index(name='new_customers')

# Web traffic by year
web['year'] = web['date'].dt.year
web_yr = web.groupby('year').agg(
    sessions=('sessions', 'sum'),
    visitors=('unique_visitors', 'sum')
).reset_index()

# Merge
funnel = annual_full[['year','revenue']].merge(acq, left_on='year', right_on='signup_year', how='left') \
    .merge(web_yr, on='year', how='left')
print(funnel.round(0).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

axes[0].plot(funnel['year'], funnel['sessions']/1e6, marker='o', color=COLORS['navy'], lw=2)
axes[0].set_ylabel('Sessions (trieu)')
axes[0].set_title('Web traffic theo nam', loc='left', fontsize=11)
axes[0].annotate(f'+62% tu 2013 den 2022', xy=(2021, 11), fontsize=9, color=COLORS['forest'])

axes[1].plot(funnel['year'], funnel['new_customers']/1000, marker='o', color=COLORS['forest'], lw=2)
axes[1].set_ylabel('Signup moi (nghin)')
axes[1].set_title('Khach hang dang ky moi theo nam', loc='left', fontsize=11)
axes[1].annotate(f'Tang deu, khong sap', xy=(2021, 19), fontsize=9, color=COLORS['forest'])

axes[2].plot(funnel['year'], funnel['revenue']/1e9, marker='o', color=COLORS['brick'], lw=2)
axes[2].set_ylabel('Doanh thu (ty VND)')
axes[2].set_title('Doanh thu theo nam', loc='left', fontsize=11)
axes[2].axvspan(2018.5, 2019.5, alpha=0.15, color=COLORS['brick'])
axes[2].annotate('Sap 2019', xy=(2019, 1.4), fontsize=9, color=COLORS['brick'])
axes[2].set_xlabel('Nam')
axes[2].set_xticks(funnel['year'])

fig.suptitle('Phan ky giua dau phieu va doanh thu', fontweight='bold', y=0.995, x=0.05, ha='left')
fig.text(0.01, -0.01, 'Nguon: web_traffic.csv, customers.csv, sales.csv', fontsize=8, color=COLORS['grey'])
plt.tight_layout()
plt.show()

**Phát hiện 1.2 - Phân kỳ ba đường:**

- Web traffic tăng từ 6.8 triệu sessions (2013) lên 11.0 triệu (2022), tăng 62%.
- Signup mới tăng từ 957 (2012) lên 21.103 (2022), tăng 22 lần.
- Doanh thu giảm 39% trong cùng giai đoạn 2018-2019, không hồi phục.

Kết luận: cú sập doanh thu 2019 không phải do thiếu cầu hay thiếu khách hàng mới. Đầu phễu vẫn khoẻ. Vấn đề nằm ở **chuyển đổi và chất lượng giao dịch**: traffic biến thành đơn hàng kém hiệu quả hơn, hoặc đơn hàng mất biên lợi nhuận. Đây là hint định hướng Act 2.

### 1.3 Tính mùa vụ - cỗ máy Q2

In [ ]:
sales['quarter'] = sales['Date'].dt.quarter
sales['month']   = sales['Date'].dt.month

# Quarterly contribution
q_contrib = sales.groupby('quarter').agg(
    revenue=('Revenue','sum'),
    profit =('profit','sum')
)
q_contrib['rev_share']    = q_contrib['revenue']/q_contrib['revenue'].sum()*100
q_contrib['profit_share'] = q_contrib['profit']/q_contrib['profit'].sum()*100
q_contrib['margin']       = q_contrib['profit']/q_contrib['revenue']*100

print('Phan bo theo quy:')
print(q_contrib.round(2).to_string())

# Monthly daily average
monthly_daily = sales.groupby('month')['Revenue'].mean()
print(f'\nMay/Dec daily revenue ratio: {monthly_daily[5]/monthly_daily[12]:.2f}x')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: Quarterly contribution (revenue vs profit share)
quarters = ['Q1','Q2','Q3','Q4']
x = np.arange(4)
w = 0.35
b1 = axes[0].bar(x - w/2, q_contrib['rev_share'], w, color=COLORS['navy'], label='% Doanh thu')
b2 = axes[0].bar(x + w/2, q_contrib['profit_share'], w, color=COLORS['gold'], label='% Loi nhuan')
axes[0].set_xticks(x)
axes[0].set_xticklabels(quarters)
axes[0].set_ylabel('Ty trong (%)')
axes[0].set_title('Q2 dong gop khong can xung: 36% doanh thu, 45% loi nhuan', loc='left', fontsize=11)
axes[0].legend()
for bar in b1: axes[0].annotate(f'{bar.get_height():.1f}%', (bar.get_x()+bar.get_width()/2, bar.get_height()+0.5), ha='center', fontsize=8)
for bar in b2: axes[0].annotate(f'{bar.get_height():.1f}%', (bar.get_x()+bar.get_width()/2, bar.get_height()+0.5), ha='center', fontsize=8)

# Right: Monthly daily average
axes[1].bar(range(1,13), monthly_daily/1e6, color=[COLORS['gold'] if m in [4,5,6] else COLORS['lightgrey'] for m in range(1,13)])
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(['T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12'])
axes[1].set_ylabel('Doanh thu trung binh ngay (trieu VND)')
axes[1].set_title(f'Thang 5 cao gap {monthly_daily[5]/monthly_daily[12]:.1f}x thang 12', loc='left', fontsize=11)
axes[1].axhline(monthly_daily[12]/1e6, color=COLORS['brick'], linestyle=':', alpha=0.6)
axes[1].annotate('Thang 12 (day)', xy=(11.5, monthly_daily[12]/1e6+0.1), fontsize=8, color=COLORS['brick'])

fig.suptitle('Tinh mua vu: co may Q2 van hoat dong dung', fontweight='bold', y=1.02, x=0.05, ha='left')
fig.text(0.01, -0.02, 'Nguon: sales.csv, gop tat ca cac nam 2012-2022', fontsize=8, color=COLORS['grey'])
plt.tight_layout()
plt.show()

**Phát hiện 1.3 - Cỗ máy mùa vụ vẫn khoẻ:**

- Q2 chiếm 36% doanh thu nhưng đóng góp 45% lợi nhuận. Biên lợi nhuận Q2 đạt 17.20%, cao nhất trong năm.
- Q4 chiếm 17% doanh thu nhưng chỉ đóng góp 16% lợi nhuận. Biên Q4 dưới mức trung bình.
- Tháng 5 là đỉnh, doanh thu trung bình ngày gấp 2.6 lần tháng 12.

Hiện tượng này không bị phá vỡ bởi cú sập 2019. Cấu trúc mùa vụ vẫn nguyên vẹn, chỉ scale tổng thể bị nén lại. Nói cách khác, ngày Q2 năm 2022 vẫn cao hơn ngày Q4 năm 2017. Doanh nghiệp chưa mất khả năng bán hàng theo mùa, mới mất khả năng giữ biên.

### Kết luận Act 1

Phạm vi vấn đề đã được khoanh vùng:

| Khu vực | Tình trạng |
|---|---|
| Đầu phễu (traffic, signup) | Khoẻ, tăng đều |
| Cấu trúc mùa vụ (Q2 peak) | Khoẻ, nguyên vẹn |
| Doanh thu tổng (2019+) | Hỏng, giảm 39%, không hồi phục |
| Biên lợi nhuận | Bất ổn, biến động 9.8%-16.6% |

Vấn đề nằm ở phần giữa: từ traffic xuống đơn hàng, từ đơn hàng xuống lợi nhuận. Act 2 sẽ truy nguyên cơ chế.

---

## Act 2 - Cơ chế: Vòng xoáy Tồn kho → Giảm giá → Mất biên

Giả thuyết của Act 2:

> Doanh nghiệp nhập hàng nhanh hơn tốc độ thị trường tiêu thụ. Tồn kho tích tụ buộc giảm giá sâu để giải phóng kho. Streetwear, danh mục có biên lợi nhuận thấp nhất, không gánh nổi mức giảm giá cần thiết. Kết quả: mỗi đợt khuyến mãi đốt thêm vốn, tạo áp lực tăng tồn kho cho kỳ tiếp theo.

Bốn bước kiểm chứng:

1. Tồn kho có thực sự tích tụ không? (DOS, STR theo thời gian)
2. Mức giảm giá thực tế đã đẩy biên Streetwear âm chưa? (margin curve)
3. Hậu quả tài chính của vòng xoáy này là bao nhiêu? (frozen capital)
4. Tất cả danh mục đều mất biên trên KM, hay chỉ Streetwear? (cross-category check)

### 2.1 Tồn kho leo thang - DOS theo danh mục qua các năm

In [ ]:
inv_yr = inv.groupby(['year','category']).agg(
    avg_dos = ('days_of_supply', 'mean'),
    avg_str = ('sell_through_rate', 'mean'),
    total_stock = ('stock_on_hand', 'sum'),
    total_received = ('units_received', 'sum'),
    total_sold = ('units_sold', 'sum'),
).reset_index()

# Pivot for plotting
dos_pivot = inv_yr.pivot(index='year', columns='category', values='avg_dos')
str_pivot = inv_yr.pivot(index='year', columns='category', values='avg_str')
print('DOS trung binh theo nam x danh muc:')
print(dos_pivot.round(0).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: DOS evolution
for cat in ['Streetwear','Outdoor','GenZ','Casual']:
    axes[0].plot(dos_pivot.index, dos_pivot[cat], marker='o', lw=2, color=CAT_COLORS[cat], label=cat)
axes[0].set_xlabel('Nam')
axes[0].set_ylabel('Days of Supply (DOS), trung binh')
axes[0].set_title('DOS leo thang - ton kho mat thanh khoan', loc='left', fontsize=11)
axes[0].axhline(180, color=COLORS['brick'], linestyle=':', alpha=0.5)
axes[0].annotate('Nguong canh bao 180 ngay', xy=(2013, 200), fontsize=8, color=COLORS['brick'])
axes[0].legend(loc='upper left', fontsize=9)

# Right: STR evolution
for cat in ['Streetwear','Outdoor','GenZ','Casual']:
    axes[1].plot(str_pivot.index, str_pivot[cat]*100, marker='o', lw=2, color=CAT_COLORS[cat], label=cat)
axes[1].set_xlabel('Nam')
axes[1].set_ylabel('Sell-through Rate (%)')
axes[1].set_title('STR giam dan - hang khong duoc tieu thu kip', loc='left', fontsize=11)
axes[1].axhline(20, color=COLORS['brick'], linestyle=':', alpha=0.5)
axes[1].annotate('Nguong khoe 20%', xy=(2013, 21), fontsize=8, color=COLORS['brick'])
axes[1].legend(loc='upper right', fontsize=9)

fig.suptitle('Hai chi so ton kho cot loi: DOS tang, STR giam', fontweight='bold', y=1.02, x=0.05, ha='left')
fig.text(0.01, -0.02, 'Nguon: inventory.csv (60.247 snapshots)', fontsize=8, color=COLORS['grey'])
plt.tight_layout()
plt.show()

**Phát hiện 2.1 - Tồn kho mất thanh khoản:**

- Streetwear DOS: 184 ngày (2012) leo lên 1.730 ngày (2022). Tăng 9.4 lần.
- Outdoor DOS: 118 ngày (2012) leo lên 1.998 ngày (2022). Tăng 16.9 lần.
- STR Streetwear giảm từ 0.22 xuống 0.15. STR Outdoor giảm từ 0.27 xuống 0.10.
- Casual và GenZ có DOS cao nhưng volume nhỏ, không phải tâm bão.

DOS gần 2.000 ngày đồng nghĩa với việc nếu doanh nghiệp dừng nhập hàng từ hôm nay, kho đủ bán cho 5.5 năm tới. Đây là dấu hiệu rõ ràng của tình trạng overstock kéo dài. Stockout không phải vấn đề: số ngày hết hàng trong dữ liệu rất nhỏ, ổn định.

### 2.2 Đường cong biên Streetwear theo độ sâu giảm giá

Bước này kiểm chứng giả thuyết: liệu giảm giá có thực sự đẩy biên Streetwear âm?

In [ ]:
# Join order_items with products
oi_p = oi.merge(products[['product_id','price','cogs','category']], on='product_id', how='left')

# Total discount = (list price - actual realized price) / list price
# unit_price in order_items is the actual sale price; discount_amount is additional reduction
oi_p['list_revenue']  = oi_p['quantity'] * oi_p['price']
oi_p['line_revenue']  = oi_p['quantity'] * oi_p['unit_price'] - oi_p['discount_amount']
oi_p['line_cogs']     = oi_p['quantity'] * oi_p['cogs']
oi_p['line_profit']   = oi_p['line_revenue'] - oi_p['line_cogs']
oi_p['total_disc_pct']= (oi_p['list_revenue'] - oi_p['line_revenue']) / oi_p['list_revenue'] * 100
oi_p['has_promo']     = oi_p['promo_id'].notna()
oi_p['has_stack']     = oi_p['promo_id_2'].notna()

# Streetwear margin curve
sw = oi_p[oi_p['category']=='Streetwear'].copy()
bins   = [-100, 0, 5, 10, 15, 20, 25, 30, 40, 100]
labels = ['<0%','0-5%','5-10%','10-15%','15-20%','20-25%','25-30%','30-40%','>40%']
sw['disc_bucket'] = pd.cut(sw['total_disc_pct'], bins=bins, labels=labels, include_lowest=True)

sw_curve = sw.groupby('disc_bucket').agg(
    revenue=('line_revenue','sum'),
    profit =('line_profit','sum'),
    n_lines=('line_revenue','count')
).reset_index()
sw_curve['margin_pct'] = sw_curve['profit']/sw_curve['revenue']*100

print('Streetwear: bien loi nhuan theo do sau giam gia thuc te')
print(sw_curve.round(2).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

colors = [COLORS['forest'] if m > 0 else COLORS['brick'] for m in sw_curve['margin_pct']]
bars = ax.bar(sw_curve['disc_bucket'].astype(str), sw_curve['margin_pct'], color=colors, edgecolor='white')

# Breakeven line
ax.axhline(0, color='black', lw=1)
# Annotate breakeven
ax.axvspan(3.5, 4.5, alpha=0.15, color=COLORS['gold'])
ax.annotate('Vung breakeven\n(15-20% discount)', xy=(4, 8),
            fontsize=9, color=COLORS['grey'], ha='center')

# Value labels
for bar, val in zip(bars, sw_curve['margin_pct']):
    y_offset = 1.5 if val >= 0 else -3
    ax.annotate(f'{val:.1f}%',
                xy=(bar.get_x()+bar.get_width()/2, val + y_offset),
                ha='center', fontsize=9,
                color='black')

ax.set_xlabel('Tong muc giam gia thuc te (%)')
ax.set_ylabel('Bien loi nhuan (%)')
ax.set_title('Streetwear: bien loi nhuan sup ngay khi giam gia vuot 20%',
             loc='left', fontweight='bold', fontsize=12)
ax.set_ylim(-75, 30)
fig.text(0.01, -0.02, f'Nguon: order_items.csv x products.csv, n={len(sw):,} dong Streetwear', fontsize=8, color=COLORS['grey'])
plt.tight_layout()
plt.show()

**Phát hiện 2.2 - Ngưỡng breakeven 20%:**

| Vùng giảm giá | Biên lợi nhuận | Ý nghĩa |
|---|---|---|
| 0-5% | 18.5% | Vùng khoẻ |
| 10-15% | 5.6% | Còn an toàn |
| 15-20% | **1.2%** | Sát breakeven |
| 20-25% | -3.3% | Vùng âm |
| 30-40% | -22.0% | Phá huỷ vốn |
| >40% | -63.3% | Lỗ nặng |

Streetwear có dư địa giảm giá đến 20%. Vượt qua ngưỡng này, mỗi đơn hàng khuyến mãi đều bán dưới giá vốn. 2/3 khối lượng Streetwear khuyến mãi rơi vào vùng giảm giá 20-40%, đó là nguồn gốc của lỗ luỹ kế.

### 2.3 Mọi danh mục đều âm biên trên khuyến mãi

Hỏi tiếp: nếu tách riêng các đơn có khuyến mãi, danh mục nào đang lỗ?

In [ ]:
promo_margin = oi_p[oi_p['has_promo']].groupby('category').agg(
    revenue=('line_revenue','sum'),
    profit =('line_profit','sum'),
    n_lines=('line_revenue','count')
).reset_index()
promo_margin['margin_pct'] = promo_margin['profit']/promo_margin['revenue']*100
promo_margin['loss_M']     = -promo_margin['profit']/1e6  # negative profit -> positive loss

overall_margin = oi_p.groupby('category').agg(
    revenue=('line_revenue','sum'),
    profit =('line_profit','sum')
).reset_index()
overall_margin['margin_pct'] = overall_margin['profit']/overall_margin['revenue']*100

compare = overall_margin[['category','margin_pct']].rename(columns={'margin_pct':'all_items'}) \
    .merge(promo_margin[['category','margin_pct','loss_M']].rename(columns={'margin_pct':'promo_items'}), on='category')
print('Bien loi nhuan: tat ca don vs don co khuyen mai')
print(compare.round(2).to_string(index=False))
print(f'\nTong lo trong KM (4 danh muc): {compare["loss_M"].sum():.1f} trieu VND')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))

cats = ['Streetwear','Outdoor','Casual','GenZ']
x = np.arange(len(cats))
w = 0.35

all_margin = compare.set_index('category').loc[cats]['all_items']
promo_m    = compare.set_index('category').loc[cats]['promo_items']

ax.bar(x - w/2, all_margin, w, color=COLORS['navy'], label='Tat ca don hang')
ax.bar(x + w/2, promo_m, w, color=COLORS['brick'], label='Don co khuyen mai')

for i, (a, p) in enumerate(zip(all_margin, promo_m)):
    ax.annotate(f'{a:+.1f}%', (i-w/2, a + (1 if a>=0 else -2)), ha='center', fontsize=9)
    ax.annotate(f'{p:+.1f}%', (i+w/2, p + (1 if p>=0 else -2)), ha='center', fontsize=9)

ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(cats)
ax.set_ylabel('Bien loi nhuan (%)')
ax.set_title('Khuyen mai am bien o moi danh muc, Streetwear thiet hai lon nhat',
             loc='left', fontweight='bold', fontsize=12)
ax.legend(loc='lower left')
fig.text(0.01, -0.02, 'Nguon: order_items.csv x products.csv', fontsize=8, color=COLORS['grey'])
plt.tight_layout()
plt.show()

**Phát hiện 2.3 - Mọi danh mục đều mất biên trên khuyến mãi:**

| Danh mục | Biên toàn bộ đơn | Biên đơn KM | Lỗ luỹ kế trên KM |
|---|---|---|---|
| Streetwear | +9.3% | -15.6% | 578.7 triệu VND |
| Outdoor | +11.4% | -9.9% | 75.6 triệu VND |
| Casual | +7.7% | -15.5% | 18.4 triệu VND |
| GenZ | +15.5% | -5.8% | 4.9 triệu VND |

Tổng lỗ trên các đơn có khuyến mãi: **677.6 triệu VND**. Streetwear chiếm 85% mức lỗ này do volume cực lớn.

Phát hiện này phủ định một giả định ngầm trong notebook trước: rằng Outdoor và GenZ "an toàn" trên khuyến mãi. Thực tế cả bốn danh mục đều âm biên khi giảm giá. Streetwear chỉ là nạn nhân lớn nhất do đóng góp 80% volume.

### 2.4 Định lượng vốn bị đóng băng

In [ ]:
# Liquidation list: SKUs with DOS > 180 in last 3 months
last_date = inv['snapshot_date'].max()
last3 = inv[inv['snapshot_date'] >= last_date - pd.Timedelta(days=92)]

sku_summary = last3.groupby(['product_id','category']).agg(
    avg_dos   = ('days_of_supply', 'mean'),
    avg_stock = ('stock_on_hand', 'mean'),
).reset_index()
liq = sku_summary[sku_summary['avg_dos'] > 180].merge(
    products[['product_id','cogs','price']], on='product_id'
)
liq['frozen_capital'] = liq['avg_stock'] * liq['cogs']

frozen_by_cat = liq.groupby('category').agg(
    n_skus=('product_id','count'),
    frozen=('frozen_capital','sum')
).reset_index().sort_values('frozen', ascending=False)
frozen_by_cat['frozen_M'] = frozen_by_cat['frozen']/1e6

total_frozen = liq['frozen_capital'].sum()
print(f'Tong SKUs DOS > 180 ngay (3 thang gan nhat): {len(liq):,}')
print(f'Tong von dong bang: {total_frozen/1e6:.1f} trieu VND')
print(f'\nPhan bo theo danh muc:')
print(frozen_by_cat.round(1).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

cats_sorted = frozen_by_cat['category'].tolist()
values = frozen_by_cat['frozen_M'].tolist()
colors = [CAT_COLORS[c] for c in cats_sorted]

bars = ax.barh(cats_sorted, values, color=colors)
for bar, val, n in zip(bars, values, frozen_by_cat['n_skus']):
    ax.annotate(f'{val:.1f}M VND ({n} SKUs)',
                (val + 5, bar.get_y() + bar.get_height()/2),
                va='center', fontsize=9)

ax.set_xlabel('Von dong bang (trieu VND)')
ax.set_title(f'Von luu dong bi dong bang: {total_frozen/1e6:.0f} trieu VND tren {len(liq)} SKUs',
             loc='left', fontweight='bold', fontsize=12)
ax.set_xlim(0, max(values)*1.3)
fig.text(0.01, -0.05, 'Nguon: inventory.csv (3 thang gan nhat) x products.csv. Tieu chi: DOS > 180 ngay',
         fontsize=8, color=COLORS['grey'])
plt.tight_layout()
plt.show()

**Phát hiện 2.4 - Quy mô vốn đóng băng:**

- 379 SKUs có DOS trên 180 ngày trong 3 tháng gần nhất, đại diện cho 471 triệu VND vốn lưu động bị đóng băng (giá trị tính theo COGS).
- Streetwear chiếm 187 SKUs / 356M VND (76% tổng frozen capital).
- Outdoor chiếm 120 SKUs / 90M VND.

Đây là số tiền có thể huy động được nếu thanh lý quyết liệt (chấp nhận giảm giá sâu một lần) thay vì kéo dài chu kỳ giảm giá nhỏ giọt năm này qua năm khác.

### Kết luận Act 2

Vòng xoáy đã được xác nhận có 4 mắt xích:

1. Nhập hàng vượt cầu, tích tụ tồn kho qua 10 năm. DOS Streetwear leo từ 184 lên 1.730 ngày.
2. Áp lực giải phóng kho buộc giảm giá. Khoảng 50% volume Streetwear khuyến mãi vượt ngưỡng 20%.
3. Vượt 20% là vùng âm biên. Lỗ luỹ kế trên Streetwear khuyến mãi: 578.7 triệu VND.
4. Lỗ làm cạn vốn, hạn chế khả năng đổi sản phẩm mới, giảm sức hấp dẫn, tồn kho càng tích.

471 triệu VND đang chôn trong kho. Câu hỏi Act 3: ai thiết kế ra các chương trình giảm giá đẩy Streetwear vượt 20%?

---

## Act 3 - Đòn bẩy: Kiến trúc khuyến mãi sai chỗ

Act 2 cho thấy giảm giá là cơ chế gây tổn thương. Act 3 truy nguyên ai thiết kế ra cơ chế đó. Ba câu hỏi:

1. Khuyến mãi hiện tại được thiết kế như thế nào (theo danh mục, theo độ sâu)?
2. Tại sao Streetwear bị kéo vào vùng giảm sâu?
3. Có pattern khuyến mãi nào đang hoạt động đúng đáng được nhân rộng?

### 3.1 80% chương trình áp dụng toàn sàn

In [ ]:
# Promo design analysis
promo_design = promos.copy()
promo_design['scope'] = promo_design['applicable_category'].fillna('All Categories (blanket)')
scope_counts = promo_design['scope'].value_counts()
print('Phan bo chuong trinh khuyen mai:')
print(scope_counts.to_string())
print(f'\nTy le blanket: {scope_counts["All Categories (blanket)"]/len(promo_design)*100:.0f}%')

# Discount depth by promo_type
print('\nMuc giam gia (percentage):')
pct_promos = promo_design[promo_design['promo_type']=='percentage']
print(pct_promos['discount_value'].describe().round(2).to_string())

# Quarterly distribution of promos
promo_design['start_quarter'] = promo_design['start_date'].dt.quarter
promo_design['discount_label'] = np.where(
    promo_design['promo_type']=='percentage',
    promo_design['discount_value'].astype(str)+'%',
    promo_design['discount_value'].astype(str)+' VND')
print('\nMuc giam gia trung binh (% only) theo quy bat dau:')
print(pct_promos.assign(q=pct_promos['start_date'].dt.quarter).groupby('q')['discount_value'].agg(['mean','count','min','max']).round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: Scope breakdown
scope_data = pd.DataFrame({
    'scope': ['Toan san (blanket)', 'Streetwear rieng', 'Outdoor rieng'],
    'count': [40, 5, 5]
})
colors_pie = [COLORS['brick'], CAT_COLORS['Streetwear'], CAT_COLORS['Outdoor']]
axes[0].pie(scope_data['count'], labels=scope_data['scope'], colors=colors_pie,
            autopct='%1.0f%%', startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('80% chuong trinh khuyen mai ap dung toan san',
                   loc='left', fontweight='bold', fontsize=11)

# Right: Discount depth distribution by quarter
quarters_q = pct_promos['start_date'].dt.quarter
disc_by_q = pct_promos.groupby(quarters_q)['discount_value'].apply(list)
positions = [1,2,3,4]
data_to_plot = [disc_by_q.get(q, []) for q in [1,2,3,4]]
bp = axes[1].boxplot(data_to_plot, positions=positions, patch_artist=True, widths=0.6)
for patch, q in zip(bp['boxes'], [1,2,3,4]):
    patch.set_facecolor(COLORS['gold'] if q==2 else COLORS['lightgrey'])
axes[1].axhline(20, color=COLORS['brick'], linestyle=':', alpha=0.7)
axes[1].annotate('Nguong breakeven Streetwear (20%)', xy=(2.5, 20.5), fontsize=8, color=COLORS['brick'])
axes[1].set_xticks(positions)
axes[1].set_xticklabels(['Q1','Q2','Q3','Q4'])
axes[1].set_ylabel('Muc giam gia (%)')
axes[1].set_title('Q4 dam vao nguong breakeven thuong xuyen hon Q2',
                   loc='left', fontweight='bold', fontsize=11)

fig.suptitle('Kien truc khuyen mai: blanket diem rong, Q4 giam sau', fontweight='bold', y=1.02, x=0.05, ha='left')
fig.text(0.01, -0.02, 'Nguon: promotions.csv (n=50 promo, 45 percentage)', fontsize=8, color=COLORS['grey'])
plt.tight_layout()
plt.show()

**Phát hiện 3.1 - Kiến trúc khuyến mãi sai về thiết kế:**

- 40/50 chương trình (80%) áp dụng toàn sàn, không phân biệt biên lợi nhuận từng danh mục.
- 5 chương trình thiết kế riêng cho Streetwear, 5 cho Outdoor.
- Mức giảm giá theo quý: Q2 trung bình ~14%, Q4 trung bình ~17%, max 20%.
- Q2 hoạt động đúng vì rất ít chương trình chạm 20%. Q4 thường đẩy lên gần ngưỡng.

Hệ quả: mỗi đợt blanket sale, Streetwear bị "cuốn" vào discount ≥15%. Khi cộng thêm các yếu tố khác (giá tăng tự nhiên do làm tròn, stack với promo nhỏ), tổng discount thực tế chạm vùng 20-30%. Đây là vùng âm biên Streetwear.

### 3.2 Nghịch lý cộng dồn (Stacking Paradox)

Trong order_items có hai cột promo: `promo_id` và `promo_id_2`. Đơn cộng dồn lý thuyết phải giảm sâu hơn, ăn biên nhiều hơn. Thực tế ngược lại.

In [ ]:
# Stacking analysis
solo    = oi_p[oi_p['has_promo'] & ~oi_p['has_stack']]
stacked = oi_p[oi_p['has_stack']]
no_promo= oi_p[~oi_p['has_promo']]

stack_summary = pd.DataFrame({
    'group': ['Khong KM', 'KM don le', 'KM cong don'],
    'n_lines': [len(no_promo), len(solo), len(stacked)],
    'margin_pct': [
        no_promo['line_profit'].sum()/no_promo['line_revenue'].sum()*100,
        solo['line_profit'].sum()/solo['line_revenue'].sum()*100,
        stacked['line_profit'].sum()/stacked['line_revenue'].sum()*100,
    ],
    'loss_rate_pct': [
        (no_promo['line_profit']<0).mean()*100,
        (solo['line_profit']<0).mean()*100,
        (stacked['line_profit']<0).mean()*100,
    ]
})

# Basket analysis
basket_solo = solo.groupby('order_id')['line_revenue'].sum()
basket_stacked = stacked.groupby('order_id')['line_revenue'].sum()
stack_summary['basket_median'] = [
    no_promo.groupby('order_id')['line_revenue'].sum().median(),
    basket_solo.median(),
    basket_stacked.median()
]

print('Bang so sanh:')
print(stack_summary.round(2).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

groups = ['Khong KM','KM don le','KM cong don']
colors_g = [COLORS['forest'], COLORS['brick'], COLORS['gold']]

# Margin
m_vals = stack_summary['margin_pct'].values
axes[0].bar(groups, m_vals, color=colors_g)
axes[0].axhline(0, color='black', lw=0.8)
for i, v in enumerate(m_vals):
    axes[0].annotate(f'{v:+.1f}%', (i, v + (1 if v>=0 else -2)), ha='center', fontsize=10)
axes[0].set_ylabel('Bien loi nhuan (%)')
axes[0].set_title('Bien loi nhuan', loc='left', fontsize=11)

# Loss rate
lr_vals = stack_summary['loss_rate_pct'].values
axes[1].bar(groups, lr_vals, color=colors_g)
for i, v in enumerate(lr_vals):
    axes[1].annotate(f'{v:.1f}%', (i, v+1), ha='center', fontsize=10)
axes[1].set_ylabel('Ty le don ban lo (%)')
axes[1].set_title('Ty le don ban duoi gia von', loc='left', fontsize=11)

# Basket median
bm_vals = stack_summary['basket_median'].values
axes[2].bar(groups, bm_vals/1000, color=colors_g)
for i, v in enumerate(bm_vals):
    axes[2].annotate(f'{v/1000:.1f}k', (i, v/1000+0.5), ha='center', fontsize=10)
axes[2].set_ylabel('Gia tri don trung vi (nghin VND)')
axes[2].set_title('Quy mo gio hang', loc='left', fontsize=11)

fig.suptitle('Nghich ly cong don: KM cong don tot hon KM don le tren ca 3 chi so',
             fontweight='bold', y=1.02, x=0.05, ha='left')
fig.text(0.01, -0.02, f'Nguon: order_items.csv. Solo n={len(solo):,}, Stacked n={len(stacked):,}', fontsize=8, color=COLORS['grey'])
plt.tight_layout()
plt.show()

**Phát hiện 3.2 - Nghịch lý cộng dồn:**

| Nhóm | Biên LN | Tỷ lệ đơn lỗ | Median basket | n |
|---|---|---|---|---|
| Không KM | +20.0% | 0.2% | (mọi range) | 438.353 |
| KM đơn lẻ | -14.5% | 69.0% | 13,4k VND | 276.110 |
| KM cộng dồn | +2.1% | 55.8% | 23,9k VND | 206 |

KM cộng dồn cải thiện biên 16.5 điểm phần trăm so với KM đơn lẻ, dù cộng thêm một lớp giảm giá. Cơ chế giải thích: 12/50 chương trình stackable đều có `min_order_value`. Điều kiện min_order_value lọc ra basket lớn hơn (median 23,9k vs 13,4k), nơi mix sản phẩm có biên cao hơn trung bình. Stacking về bản chất là một cơ chế qualification, không phải mechanism giảm giá thuần.

**Implication:** mở rộng chương trình stackable với min_order_value cao có thể giảm tỷ lệ đơn lỗ tổng thể. Đây là một đòn bẩy ít tốn kém vì chỉ thay đổi quy tắc trigger, không thay đổi mức discount.

### 3.3 Bốn hành động định lượng được

In [ ]:
recommendations = pd.DataFrame([
    {
        'STT': 1,
        'Hanh dong': 'Hard cap 20% discount cho Streetwear',
        'Co che': 'Block tai payment gateway cac don Streetwear vuot 20% discount tong',
        'Tac dong uoc tinh': '+15-20% bien Streetwear KM (tu -15.6% len 0-5%)',
        'Loi ich tai chinh': '~80M VND/nam (giam lo Streetwear 578.7M trong 11 nam, ~52M/nam)',
        'Timeline': '1-2 tuan trien khai',
        'KPI': 'Bien Streetwear KM > 0%'
    },
    {
        'STT': 2,
        'Hanh dong': 'Cat 25% recurring orders cho Streetwear/Outdoor',
        'Co che': 'Giam lenh nhap dinh ky 25% trong 12 thang',
        'Tac dong uoc tinh': 'Giai phong 280-320M VND von luu dong',
        'Loi ich tai chinh': 'Cash flow tot hon, giam ap luc giam gia tuong lai',
        'Timeline': 'Quy 1-2 (6 thang)',
        'KPI': 'DOS Streetwear < 800 ngay trong 6 thang'
    },
    {
        'STT': 3,
        'Hanh dong': 'Thanh ly 379 SKUs DOS > 180',
        'Co che': 'Mot dot giam gia sau (40-50%) cho ton kho dong, chap nhan lo ngan han',
        'Tac dong uoc tinh': 'Thu hoi 200-300M VND von',
        'Loi ich tai chinh': '60-70% von hien tai dang dong bang',
        'Timeline': 'Quy 1 (3 thang)',
        'KPI': 'Frozen capital < 200M VND'
    },
    {
        'STT': 4,
        'Hanh dong': 'Tai cau truc kien truc khuyen mai 80:20 -> 40:60',
        'Co che': 'Giam ty le blanket promo, tang category-specific (Outdoor/GenZ co bien tot)',
        'Tac dong uoc tinh': 'Giam cross-category profit erosion 10-15M VND/nam',
        'Loi ich tai chinh': 'Bao ve Streetwear khoi blanket sale dam',
        'Timeline': '12 thang chuyen doi tu tu',
        'KPI': 'Ty le blanket promo < 50% trong 1 nam'
    },
])
print(recommendations.to_string(index=False))

**Tổng tác động ước tính 12 tháng:**

| Hành động | Tác động vốn lưu động | Tác động lợi nhuận năm |
|---|---|---|
| 1. Hard cap 20% Streetwear | - | +50-80M VND |
| 2. Cắt 25% nhập định kỳ | +280-320M VND | +20-30M VND (tránh giảm giá ép) |
| 3. Thanh lý 379 SKUs | +200-300M VND | -50M VND (lỗ thanh lý ngắn hạn) |
| 4. Tái cấu trúc promo | - | +10-15M VND |
| **Tổng** | **+480-620M VND** | **+30-75M VND** |

Trade-off chính: hành động 3 phải chịu lỗ thanh lý ngắn hạn để thu hồi vốn. Tổng net positive: vốn lưu động tăng gần gấp đôi sau 12 tháng.

---

## Act 4 - Hệ luỵ cho dự báo: Vì sao baseline notebook over-forecast

EDA định hình lựa chọn mô hình. Baseline notebook giả định doanh nghiệp đang tăng trưởng dương dựa trên geometric mean YoY 2013-2022. Phân tích Act 1 cho thấy giả định này sai.

In [ ]:
# Replicate baseline approach: geometric mean YoY 2013-2022
yoy_rev = annual_full['revenue'].pct_change().dropna()
geo_mean_full = (1 + yoy_rev).prod() ** (1/len(yoy_rev))
print(f'Geometric mean YoY 2013-2022 (baseline assumption): {(geo_mean_full-1)*100:+.2f}% per year')

# Realistic regime: 2019-2022 plateau
recent = annual_full[annual_full['year']>=2019]
yoy_recent = recent['revenue'].pct_change().dropna()
geo_mean_recent = (1 + yoy_recent).prod() ** (1/len(yoy_recent))
print(f'Geometric mean YoY 2019-2022 (post-crash regime): {(geo_mean_recent-1)*100:+.2f}% per year')

# Backtest: predict 2021-2022 using baseline approach trained on 2013-2020
train = annual_full[annual_full['year']<=2020]
train_yoy = train['revenue'].pct_change().dropna()
train_geo = (1 + train_yoy).prod() ** (1/len(train_yoy))
base_2020 = train[train['year']==2020]['revenue'].values[0]

actual_2021 = annual_full[annual_full['year']==2021]['revenue'].values[0]
actual_2022 = annual_full[annual_full['year']==2022]['revenue'].values[0]
pred_2021 = base_2020 * train_geo
pred_2022 = base_2020 * train_geo**2

err_2021 = (pred_2021/actual_2021 - 1)*100
err_2022 = (pred_2022/actual_2022 - 1)*100
print(f'\n--- Backtest baseline approach trained on 2013-2020 ---')
print(f'2021 actual:    {actual_2021/1e9:.2f} ty')
print(f'2021 predicted: {pred_2021/1e9:.2f} ty  (error: {err_2021:+.1f}%)')
print(f'2022 actual:    {actual_2022/1e9:.2f} ty')
print(f'2022 predicted: {pred_2022/1e9:.2f} ty  (error: {err_2022:+.1f}%)')

**Phát hiện 4 - Baseline over-forecast 2-12%:**

- Geometric mean toàn bộ chuỗi: dương ~6%/năm (giả định tăng trưởng).
- Geometric mean post-2019: gần phẳng (~3%/năm).
- Backtest cho thấy baseline approach predict 2021-2022 bằng dữ liệu 2013-2020 sẽ over-forecast.

**Khuyến nghị mô hình hoá Phần 3:**

1. Loại bỏ giả định tăng trưởng đồng đều. Học pattern plateau từ 2019-2022, không học toàn bộ chuỗi.
2. Sử dụng features có signal causal: stockout flag, promo intensity, web sessions, day-of-week, holiday calendar.
3. Cross-validation theo chiều thời gian (TimeSeriesSplit), không random split.
4. Ensemble mô hình tuyến tính (cho trend) với gradient boosting (cho non-linear seasonal interactions).
5. Predict separately Revenue và COGS, không suy COGS từ Revenue (vì biên đang bất ổn).

---

## Tổng kết

Hai cỗ máy bên trong, một chạy đúng, một hỏng:

**Cỗ máy mùa vụ (khoẻ):** Q2 vẫn đóng góp 45% lợi nhuận năm. Tháng 5 cao gấp 2.6x tháng 12. Pattern này không bị phá vỡ bởi cú sập 2019.

**Cỗ máy lợi nhuận (hỏng):** Tồn kho Streetwear/Outdoor leo từ 150 ngày lên gần 2.000 ngày DOS. Giảm giá vượt 20% đẩy biên Streetwear âm. 471M VND vốn đóng băng. Lỗ luỹ kế trên Streetwear KM đạt 578.7M VND.

**Đòn bẩy:** Hard cap 20% Streetwear, cắt 25% nhập định kỳ, thanh lý 379 SKUs DOS cao, tái cấu trúc tỷ lệ blanket:specific từ 80:20 xuống 40:60. Tổng impact ước tính 480-620M VND vốn lưu động + 30-75M VND lợi nhuận năm.

**Hệ luỵ forecasting:** Baseline notebook over-forecast vì học geometric mean toàn chuỗi. Mô hình tốt phải học regime post-2019.

---

*Repository: [GitHub link]*
*Tác giả: [Team name]*